# Global tables by optimization objective

This notebook consolidates, for model `Llama3.1-I`, comparisons between:
- `lod`;
- `llama / initial_system_prompt` (`without_optimization` result);
- `llama / best_system_prompt` (the `with_optimization` configuration whose process reached the best training value for the analyzed objective).

The output is organized by optimization objective:
- `sep`
- `etd`
- `sep_etd_f1`

For each objective, the notebook:
1. chooses the best optimized process per algorithm based on the highest `best_train_metric` stored by the optimization process;
2. shows a summary table of the selected best configuration;
3. displays three tables with the test-set `SEP`, `ETD`, and `SEP_ETD_F1` metrics for `lod`, `initial_system_prompt`, and `best_system_prompt`;
4. annotates the `best_system_prompt` values directly with two symbols per cell, in the order `vs lod` and then `vs initial`, using a paired Wilcoxon test:
   - `▲` for better and significant,
   - `●` for no significant difference,
   - `▼` for worse and significant.


In [1]:
import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

try:
    from scipy.stats import wilcoxon
except ImportError:
    wilcoxon = None


In [2]:
def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not locate the explainability-with-LLMs root. "
        "Run the notebook inside the project, from one of its subdirectories, or from the workspace root."
    )


def display_workspace_relative(path: Path, workspace_root: Path) -> str:
    resolved = path.resolve()
    try:
        return str(resolved.relative_to(workspace_root.resolve()))
    except ValueError:
        return str(resolved)


def display_path_string(path_str: str | None, workspace_root: Path) -> str | None:
    if not path_str:
        return path_str

    path = Path(path_str)
    if not path.is_absolute():
        return path_str

    return display_workspace_relative(path, workspace_root)

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
WORKSPACE_ROOT = PROJECT_ROOT.parent
MODEL_NAME = "Llama3.1-I"
SUMMARY_CSV = PROJECT_ROOT / "analyse_results" / "statistical_analysis" / "common" / "per_user_run_summary.csv"
STATISTICAL_ANALYSIS_ROOT = PROJECT_ROOT / "analyse_results" / "statistical_analysis"
LOD_ROOT = PROJECT_ROOT.parent / "datasets" / "lod_results" / "individual_metrics"

print(f"PROJECT_ROOT: {display_workspace_relative(PROJECT_ROOT, WORKSPACE_ROOT)}")
print(f"SUMMARY_CSV: {display_workspace_relative(SUMMARY_CSV, WORKSPACE_ROOT)}")
print(
    "STATISTICAL_ANALYSIS_ROOT: "
    f"{display_workspace_relative(STATISTICAL_ANALYSIS_ROOT, WORKSPACE_ROOT)}"
)
print(f"LOD_ROOT: {display_workspace_relative(LOD_ROOT, WORKSPACE_ROOT)}")


PROJECT_ROOT: explainability-with-LLMs
SUMMARY_CSV: explainability-with-LLMs/analyse_results/statistical_analysis/common/per_user_run_summary.csv
STATISTICAL_ANALYSIS_ROOT: explainability-with-LLMs/analyse_results/statistical_analysis
LOD_ROOT: datasets/lod_results/individual_metrics


In [3]:
OBJECTIVE_ORDER = ["sep", "etd", "sep_etd_f1"]
ALGORITHM_ORDER = ["user_knn", "item_knn", "bprmf", "ncf"]
TARGET_COLUMN_BY_OBJECTIVE = {
    "sep": "mean_sep_per_user",
    "etd": "mean_etd_per_user",
    "sep_etd_f1": "mean_sep_etd_f1_per_user",
}
DISPLAY_METRICS = [
    ("mean_sep_per_user", "sep_value", "SEP", "sep"),
    ("mean_etd_per_user", "etd_value", "ETD", "etd"),
    ("mean_sep_etd_f1_per_user", "sep_etd_f1_value", "SEP_ETD_F1", "sep_etd_f1"),
]
ALPHA = 0.05
SYMBOLS = {
    "gain": "▲",
    "tie": "●",
    "loss": "▼",
    "missing": "-",
}
ROW_INDEX = [
    ("lod", ""),
    ("llama", "initial_system_prompt"),
    ("llama", "best_system_prompt"),
]


def ordered_algorithms(values: list[str]) -> list[str]:
    ordered = [algorithm for algorithm in ALGORITHM_ORDER if algorithm in values]
    ordered.extend(sorted(algorithm for algorithm in values if algorithm not in ordered))
    return ordered


def harmonic_mean(left: float, right: float) -> float:
    if pd.isna(left) or pd.isna(right) or (left + right) == 0:
        return 0.0
    return float((2.0 * left * right) / (left + right))


def wilcoxon_symbol_from_series(
    best_series: pd.Series,
    reference_series: pd.Series,
    alpha: float = ALPHA,
) -> str:
    paired = pd.DataFrame(
        {
            "best": pd.to_numeric(best_series, errors="coerce"),
            "reference": pd.to_numeric(reference_series, errors="coerce"),
        }
    ).dropna()

    if paired.empty or wilcoxon is None:
        return SYMBOLS["missing"]

    try:
        result = wilcoxon(
            paired["best"],
            paired["reference"],
            alternative="two-sided",
            zero_method="wilcox",
        )
        p_value = float(result.pvalue)
    except ValueError:
        return SYMBOLS["missing"]

    if pd.isna(p_value) or p_value >= alpha:
        return SYMBOLS["tie"]

    mean_best = float(paired["best"].mean())
    mean_reference = float(paired["reference"].mean())
    if mean_best > mean_reference:
        return SYMBOLS["gain"]
    if mean_best < mean_reference:
        return SYMBOLS["loss"]
    return SYMBOLS["tie"]


def load_summary(summary_csv: Path) -> pd.DataFrame:
    if not summary_csv.exists():
        raise FileNotFoundError(f"Statistical summary not found at {summary_csv}")

    df = pd.read_csv(summary_csv)
    numeric_cols = [
        "mean_sep_per_user",
        "mean_etd_per_user",
        "mean_sep_etd_f1_per_user",
        "n_users",
    ]
    for column in numeric_cols:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def load_process_selection_payload(process_dir: Path) -> dict[str, object]:
    metadata_path = process_dir / "optimization_process_metadata.json"
    if not metadata_path.exists():
        return {
            "best_train_metric": pd.NA,
            "best_val_metric": pd.NA,
            "time_prompt_optimization": pd.NA,
            "best_train_epoch": pd.NA,
            "best_val_epoch": pd.NA,
        }

    payload = json.loads(metadata_path.read_text())
    best_on_train = payload.get("best_on_train", {})
    best_on_validation = payload.get("best_on_validation", {})
    return {
        "best_train_metric": best_on_train.get("best_train_metric", pd.NA),
        "best_val_metric": best_on_validation.get("best_val_metric", pd.NA),
        "time_prompt_optimization": payload.get("time_prompt_optimization", pd.NA),
        "best_train_epoch": best_on_train.get("best_train_epoch", pd.NA),
        "best_val_epoch": best_on_validation.get("best_val_epoch", pd.NA),
    }


def resolve_project_relative_path(path_str: str | float | None, project_root: Path) -> Path | None:
    if path_str is None or pd.isna(path_str):
        return None

    candidate = Path(str(path_str))
    if candidate.is_absolute():
        return candidate

    for base in (project_root, project_root.parent, project_root.parent.parent):
        resolved = (base / candidate).resolve()
        if resolved.exists():
            return resolved

    return (project_root.parent.parent / candidate).resolve()


def load_lod_summary(lod_root: Path) -> pd.DataFrame:
    rows = []
    for csv_path in sorted(lod_root.glob("indiv_metrics_explanations_optimized_*_K=20_recs.csv")):
        algorithm = (
            csv_path.name
            .replace("indiv_metrics_explanations_optimized_", "")
            .replace("_K=20_recs.csv", "")
        )
        frame = pd.read_csv(csv_path)
        frame["sep"] = pd.to_numeric(frame["sep"], errors="coerce")
        frame["etd"] = pd.to_numeric(frame["etd"], errors="coerce")
        frame["sep_etd_f1"] = frame.apply(
            lambda row: harmonic_mean(row["sep"], row["etd"]),
            axis=1,
        )
        rows.append(
            {
                "algorithm": algorithm,
                "mean_sep_per_user": frame["sep"].mean(),
                "mean_etd_per_user": frame["etd"].mean(),
                "mean_sep_etd_f1_per_user": frame["sep_etd_f1"].mean(),
                "n_users": int(frame["userId"].nunique()),
                "source_group": "lod",
                "source_label": "",
                "representation_model": pd.NA,
                "mmr_lambda_tag": pd.NA,
                "run_label": f"lod/{algorithm}",
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "algorithm",
                "mean_sep_per_user",
                "mean_etd_per_user",
                "mean_sep_etd_f1_per_user",
                "n_users",
                "source_group",
                "source_label",
                "representation_model",
                "mmr_lambda_tag",
                "run_label",
            ]
        )

    return pd.DataFrame(rows)


def format_repr_model(value: str | float | None) -> str:
    if value is None or pd.isna(value):
        return "-"
    return str(value).replace("repr_", "")


def format_lambda(value: str | float | None) -> str:
    if value is None or pd.isna(value):
        return "-"
    return str(value).replace("mmr_lambda_", "").replace("_", ".")


def build_process_dir_rel(row: pd.Series) -> str:
    return (
        "out/prompt_optimization/"
        + str(row["model"])
        + "/"
        + str(row["algorithm"])
        + "/"
        + str(row["metric"])
        + "/"
        + str(row["representation_model"])
        + "/"
        + str(row["early_stopping_tag"])
        + "/"
        + str(row["mmr_lambda_tag"])
        + "/"
        + str(row["mmr_pool_tag"])
        + "/"
        + str(row["model"])
        + "/prompt_opt/"
        + str(row["metric"])
    )


def enrich_optimized_runs_with_process_selection(
    summary_df: pd.DataFrame,
    project_root: Path,
) -> pd.DataFrame:
    frame = summary_df.copy()
    mask = frame["run_type"] == "with_optimization"
    if not mask.any():
        return frame

    frame.loc[mask, "process_dir_rel"] = frame.loc[mask].apply(build_process_dir_rel, axis=1)
    cache: dict[str, dict[str, object]] = {}
    for index, row in frame.loc[mask].iterrows():
        process_dir_rel = str(row["process_dir_rel"])
        if process_dir_rel not in cache:
            cache[process_dir_rel] = load_process_selection_payload(project_root / process_dir_rel)
        payload = cache[process_dir_rel]
        for key, value in payload.items():
            frame.at[index, key] = value

    for column in [
        "best_train_metric",
        "best_val_metric",
        "time_prompt_optimization",
        "best_train_epoch",
        "best_val_epoch",
    ]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def pick_initial_rows(summary_df: pd.DataFrame, objective_metric: str) -> pd.DataFrame:
    frame = summary_df[
        (summary_df["run_type"] == "without_optimization")
        & (summary_df["metric"] == objective_metric)
    ].copy()
    frame["source_group"] = "llama"
    frame["source_label"] = "initial_system_prompt"
    return frame


def pick_best_optimized_rows(summary_df: pd.DataFrame, objective_metric: str) -> pd.DataFrame:
    frame = summary_df[
        (summary_df["run_type"] == "with_optimization")
        & (summary_df["metric"] == objective_metric)
    ].copy()

    if frame.empty:
        return frame

    frame = frame.sort_values(
        by=[
            "algorithm",
            "best_train_metric",
            "best_val_metric",
            "time_prompt_optimization",
            "run_label",
        ],
        ascending=[True, False, False, True, True],
        na_position="last",
    )
    frame = frame.drop_duplicates(subset=["algorithm"], keep="first")
    frame["source_group"] = "llama"
    frame["source_label"] = "best_system_prompt"
    return frame


def load_per_user_metrics(csv_path: Path, score_columns: dict[str, str]) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    if "userId" not in frame.columns:
        raise KeyError(f"`userId` is missing from {csv_path}")

    selected = frame[["userId", *score_columns.keys()]].copy()
    selected["userId"] = pd.to_numeric(selected["userId"], errors="coerce")
    for source_name, target_name in score_columns.items():
        selected[source_name] = pd.to_numeric(selected[source_name], errors="coerce")
        if source_name != target_name:
            selected = selected.rename(columns={source_name: target_name})
    return selected


def load_lod_per_user_metrics(lod_root: Path, algorithm: str) -> pd.DataFrame:
    csv_path = lod_root / f"indiv_metrics_explanations_optimized_{algorithm}_K=20_recs.csv"
    if not csv_path.exists():
        return pd.DataFrame()

    return load_per_user_metrics(
        csv_path=csv_path,
        score_columns={
            "sep": "sep_lod",
            "etd": "etd_lod",
            "f1": "sep_etd_f1_lod",
        },
    )


def build_best_wide_df(
    summary_df: pd.DataFrame,
    lod_root: Path,
    objective_metric: str,
    project_root: Path,
) -> pd.DataFrame:
    best_df = pick_best_optimized_rows(summary_df, objective_metric)
    initial_df = pick_initial_rows(summary_df, objective_metric)
    if best_df.empty or initial_df.empty:
        return pd.DataFrame()

    paired_frames = []
    available_algorithms = ordered_algorithms(best_df["algorithm"].dropna().astype(str).unique().tolist())
    for algorithm in available_algorithms:
        best_rows = best_df.loc[best_df["algorithm"] == algorithm]
        initial_rows = initial_df.loc[initial_df["algorithm"] == algorithm]
        if best_rows.empty or initial_rows.empty:
            continue

        best_row = best_rows.iloc[0]
        initial_row = initial_rows.iloc[0]
        best_csv = resolve_project_relative_path(best_row.get("per_user_csv"), project_root)
        initial_csv = resolve_project_relative_path(initial_row.get("per_user_csv"), project_root)
        if best_csv is None or initial_csv is None or not best_csv.exists() or not initial_csv.exists():
            continue

        best_metrics = load_per_user_metrics(
            csv_path=best_csv,
            score_columns={
                "sep": "sep_llama_with_optimization",
                "etd": "etd_llama_with_optimization",
                "sep_etd_f1": "sep_etd_f1_llama_with_optimization",
            },
        )
        initial_metrics = load_per_user_metrics(
            csv_path=initial_csv,
            score_columns={
                "sep": "sep_llama_without_optimization",
                "etd": "etd_llama_without_optimization",
                "sep_etd_f1": "sep_etd_f1_llama_without_optimization",
            },
        )
        lod_metrics = load_lod_per_user_metrics(lod_root=lod_root, algorithm=str(algorithm))
        if lod_metrics.empty:
            continue

        paired = (
            lod_metrics.merge(initial_metrics, on="userId", how="inner")
            .merge(best_metrics, on="userId", how="inner")
            .sort_values("userId")
            .reset_index(drop=True)
        )
        if paired.empty:
            continue

        paired.insert(0, "algorithm", str(algorithm))
        paired_frames.append(paired)

    if not paired_frames:
        return pd.DataFrame()

    return pd.concat(paired_frames, ignore_index=True)


def build_best_wide_by_objective(
    summary_df: pd.DataFrame,
    lod_root: Path,
    project_root: Path,
) -> dict[str, pd.DataFrame]:
    return {
        objective_metric: build_best_wide_df(
            summary_df=summary_df,
            lod_root=lod_root,
            objective_metric=objective_metric,
            project_root=project_root,
        )
        for objective_metric in OBJECTIVE_ORDER
    }


def build_objective_comparison_df(
    summary_df: pd.DataFrame,
    lod_df: pd.DataFrame,
    objective_metric: str,
) -> pd.DataFrame:
    frames = []

    initial_df = pick_initial_rows(summary_df, objective_metric)
    if not initial_df.empty:
        frames.append(initial_df)

    best_df = pick_best_optimized_rows(summary_df, objective_metric)
    if not best_df.empty:
        frames.append(best_df)

    if not lod_df.empty:
        lod_view = lod_df.copy()
        lod_view["metric"] = objective_metric
        frames.append(lod_view)

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined["objective_metric"] = objective_metric
    combined["algorithm"] = pd.Categorical(
        combined["algorithm"],
        categories=ordered_algorithms(combined["algorithm"].dropna().astype(str).unique().tolist()),
        ordered=True,
    )
    return combined.sort_values(by=["algorithm", "source_group", "source_label"]).reset_index(drop=True)


def build_metric_pivot(
    comparison_df: pd.DataFrame,
    metric_column: str,
    top_label: str,
) -> pd.DataFrame:
    pivot = comparison_df.pivot_table(
        index=["source_group", "source_label"],
        columns="algorithm",
        values=metric_column,
        aggfunc="first",
        observed=False,
    )
    pivot = pivot.reindex(
        index=pd.MultiIndex.from_tuples(ROW_INDEX, names=[None, None])
    )
    algorithms = ordered_algorithms([str(column) for column in pivot.columns.tolist()])
    pivot = pivot.reindex(columns=algorithms)
    pivot.columns = pd.MultiIndex.from_product([[top_label], pivot.columns])
    return pivot


def build_significance_symbol_map(
    best_wide_df: pd.DataFrame,
    metric_key: str,
) -> dict[str, dict[str, str]]:
    if best_wide_df.empty:
        return {}

    best_column = f"{metric_key}_llama_with_optimization"
    initial_column = f"{metric_key}_llama_without_optimization"
    lod_column = f"{metric_key}_lod"
    required_columns = {"algorithm", best_column, initial_column, lod_column}
    if not required_columns.issubset(best_wide_df.columns):
        return {}

    available_algorithms = ordered_algorithms(
        best_wide_df["algorithm"].dropna().astype(str).unique().tolist()
    )
    symbols_by_algorithm = {}

    for algorithm in available_algorithms:
        symbols_by_algorithm[algorithm] = {
            "best_vs_initial": wilcoxon_symbol_from_series(
                best_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, best_column],
                reference_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, initial_column],
            ),
            "best_vs_lod": wilcoxon_symbol_from_series(
                best_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, best_column],
                reference_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, lod_column],
            )
        }
    return symbols_by_algorithm


def build_annotated_metric_table(
    metric_table: pd.DataFrame,
    best_wide_df: pd.DataFrame,
    metric_key: str,
    top_label: str,
) -> pd.DataFrame:
    if metric_table.empty:
        return pd.DataFrame()

    values_df = metric_table[top_label].copy()
    annotated_df = values_df.copy().astype(object)
    for row_index in annotated_df.index:
        for algorithm in annotated_df.columns:
            value = values_df.loc[row_index, algorithm]
            annotated_df.loc[row_index, algorithm] = "-" if pd.isna(value) else f"{float(value):.6f}"

    symbols_by_algorithm = build_significance_symbol_map(
        best_wide_df=best_wide_df,
        metric_key=metric_key,
    )

    best_index = ("llama", "best_system_prompt")
    if best_index in annotated_df.index:
        for algorithm in annotated_df.columns:
            base_value = annotated_df.loc[best_index, algorithm]
            if base_value == "-":
                continue

            symbol_info = symbols_by_algorithm.get(
                str(algorithm),
                {"best_vs_initial": SYMBOLS["missing"], "best_vs_lod": SYMBOLS["missing"]},
            )
            best_vs_lod = symbol_info["best_vs_lod"]
            best_vs_initial = symbol_info["best_vs_initial"]
            if best_vs_initial == SYMBOLS["missing"] and best_vs_lod == SYMBOLS["missing"]:
                continue

            annotated_df.loc[best_index, algorithm] = (
                f"{best_vs_lod}{best_vs_initial} {base_value}"
            )

    annotated_df.columns = pd.MultiIndex.from_product([[top_label], annotated_df.columns])
    return annotated_df


def build_best_config_table(comparison_df: pd.DataFrame) -> pd.DataFrame:
    best_df = comparison_df[comparison_df["source_label"] == "best_system_prompt"].copy()
    if best_df.empty:
        return pd.DataFrame()

    best_df["representation_model"] = best_df["representation_model"].apply(format_repr_model)
    best_df["mmr_lambda_quality"] = best_df["mmr_lambda_tag"].apply(format_lambda)

    columns = [
        "algorithm",
        "representation_model",
        "mmr_lambda_quality",
        "mean_sep_per_user",
        "mean_etd_per_user",
        "mean_sep_etd_f1_per_user",
        "run_label",
    ]
    return best_df[columns].rename(
        columns={
            "algorithm": "algorithm",
            "representation_model": "representation_model",
            "mmr_lambda_quality": "mmr_lambda_quality",
            "mean_sep_per_user": "sep_value",
            "mean_etd_per_user": "etd_value",
            "mean_sep_etd_f1_per_user": "sep_etd_f1_value",
            "run_label": "optimized_run_label",
        }
    ).reset_index(drop=True)


In [4]:
summary_df = enrich_optimized_runs_with_process_selection(
    load_summary(SUMMARY_CSV),
    PROJECT_ROOT,
)
lod_df = load_lod_summary(LOD_ROOT)
best_wide_by_objective = build_best_wide_by_objective(summary_df, LOD_ROOT, PROJECT_ROOT)

if summary_df.empty:
    warnings.warn("The per-user summary is empty. No table was built.")
else:
    display(Markdown(
        "The tables below use, for each objective, the best optimized process by algorithm "
        "selected by the highest `best_train_metric` reached during optimization."
    ))
    display(Markdown(
        "The symbols appear on the `best_system_prompt` row, in the order `vs lod` and then `vs initial`. "
        "Wilcoxon legend: `▲` = better and significant, "
        "`●` = no significant difference and `▼` = worse and significant."
    ))

    for objective_metric in OBJECTIVE_ORDER:
        comparison_df = build_objective_comparison_df(
            summary_df=summary_df,
            lod_df=lod_df,
            objective_metric=objective_metric,
        )

        if comparison_df.empty:
            display(Markdown(f"## Objective `{objective_metric}`"))
            display(Markdown("> No data found for this objective."))
            continue

        display(Markdown(f"## Objective `{objective_metric}`"))
        display(Markdown("### Best optimized configuration by algorithm"))
        best_config_df = build_best_config_table(comparison_df)
        best_wide_df = best_wide_by_objective.get(objective_metric, pd.DataFrame())
        if best_config_df.empty:
            display(Markdown("> No optimized configuration was found."))
        else:
            display(best_config_df.style.format(
                {
                    "sep_value": "{:.6f}",
                    "etd_value": "{:.6f}",
                    "sep_etd_f1_value": "{:.6f}",
                },
                na_rep="-",
            ))

        for metric_column, top_label, metric_label, metric_key in DISPLAY_METRICS:
            display(Markdown(
                f"### Table of `{metric_label}` for processes optimized for `{objective_metric}`"
            ))
            metric_table = build_metric_pivot(
                comparison_df=comparison_df,
                metric_column=metric_column,
                top_label=top_label,
            )
            annotated_metric_table = build_annotated_metric_table(
                metric_table=metric_table,
                best_wide_df=best_wide_df,
                metric_key=metric_key,
                top_label=top_label,
            )
            display(annotated_metric_table)


The tables below use, for each objective, the best optimized process by algorithm selected by the highest `best_train_metric` reached during optimization.

The symbols appear on the `best_system_prompt` row, in the order `vs lod` and then `vs initial`. Wilcoxon legend: `▲` = better and significant, `●` = no significant difference and `▼` = worse and significant.

## Objective `sep`

### Best optimized configuration by algorithm

,algorithm,representation_model,mmr_lambda_quality,sep_value,etd_value,sep_etd_f1_value,optimized_run_label
0,user_knn,llm2vec,0.0,0.685969,0.735246,0.692497,with_optimization/Llama3.1-I/user_knn/sep/repr_llm2vec/early_false/mmr_lambda_0_0/mmr_pool_10
1,item_knn,llm2vec,0.5,0.701002,0.663934,0.662004,with_optimization/Llama3.1-I/item_knn/sep/repr_llm2vec/early_false/mmr_lambda_0_5/mmr_pool_10
2,bprmf,sbert,0.0,0.706125,0.706557,0.692193,with_optimization/Llama3.1-I/bprmf/sep/repr_sbert/early_false/mmr_lambda_0_0/mmr_pool_10
3,ncf,sbert,0.5,0.730987,0.646721,0.664689,with_optimization/Llama3.1-I/ncf/sep/repr_sbert/early_false/mmr_lambda_0_5/mmr_pool_10


...

sep_value                            \
algorithm                       user_knn     item_knn        bprmf   
lod                             0.596438     0.560192     0.608610   
llama initial_system_prompt     0.658011     0.626747     0.662111   
      best_system_prompt     ▲▲ 0.692547  ▲● 0.621315  ▲▲ 0.685957   

                                          
algorithm                            ncf  
lod                             0.605711  
llama initial_system_prompt     0.628054  
      best_system_prompt     ▲▲ 0.656289

### Table of `ETD` for processes optimized for `sep_etd_f1`

etd_value                            \
algorithm                       user_knn     item_knn        bprmf   
lod                             0.509016     0.580328     0.524590   
llama initial_system_prompt     0.757377     0.757377     0.785246   
      best_system_prompt     ▲● 0.748273  ▲▲ 0.795082  ▲● 0.771311   

                                          
algorithm                            ncf  
lod                             0.559016  
llama initial_system_prompt     0.776230  
      best_system_prompt     ▲● 0.752459

### Table of `SEP_ETD_F1` for processes optimized for `sep_etd_f1`

sep_etd_f1_value                            \
algorithm                           user_knn     item_knn        bprmf   
lod                                 0.522916     0.547630     0.536263   
llama initial_system_prompt         0.689348     0.671721     0.702121   
      best_system_prompt         ▲● 0.700532  ▲● 0.678642  ▲● 0.712135   

                                          
algorithm                            ncf  
lod                             0.559040  
llama initial_system_prompt     0.675460  
      best_system_prompt     ▲● 0.686216